# A* Heuristic Comparison
This notebook compares four A* heuristics—Manhattan, Euclidean, Chebyshev, and Zero—on the same sequence of warehouse routes.

It records total path cost, nodes expanded, and execution time, then saves the results to `heuristic_comparison.csv`.

## 1. Import required modules
Keep this notebook in the project directory alongside `astar.py`, `warehouse.py`, `tasks.py`, and `map.txt`.

In [1]:
from pathlib import Path
import csv
import time
from astar import astar
from warehouse import Warehouse
from tasks import TASKS

ROOT = Path.cwd()
HEURISTICS = ["manhattan", "euclidean", "chebyshev", "zero"]

## 2. Load the warehouse map and define the route goals

In [2]:
warehouse = Warehouse(ROOT / "map.txt")
start = warehouse.start
goals = [task["position"] for task in TASKS] + [warehouse.packing]

print("Starting position:", start)
print("Route goals:", goals)

Starting position: (1, 1)
Route goals: [(1, 10), (3, 13), (8, 2), (9, 11), (10, 12)]


## 3. Run the four heuristic experiments
Each heuristic is tested on the same ordered goals. After reaching one goal, that position becomes the starting point for the next route segment.

In [3]:
rows = []

print("\n=== A* HEURISTIC COMPARISON ===")
print(f"{'Heuristic':<14}{'Route':<12}{'Path cost':<12}{'Expanded':<12}{'Time (ms)':<12}")

for name in HEURISTICS:
    total_cost = 0
    total_expanded = 0
    total_ms = 0.0
    current = start

    for goal in goals:
        t0 = time.perf_counter()
        path, expanded = astar(warehouse, current, goal, name)
        elapsed = (time.perf_counter() - t0) * 1000

        if not path:
            raise RuntimeError(f"No path for {name}: {current} -> {goal}")

        cost = len(path) - 1
        total_cost += cost
        total_expanded += expanded
        total_ms += elapsed
        current = goal

    rows.append({
        "heuristic": name,
        "total_path_cost": total_cost,
        "nodes_expanded": total_expanded,
        "execution_time_ms": round(total_ms, 4),
    })
    print(f"{name:<14}{'all goals':<12}{total_cost:<12}{total_expanded:<12}{total_ms:<12.4f}")


=== A* HEURISTIC COMPARISON ===
Heuristic     Route       Path cost   Expanded    Time (ms)   
manhattan     all goals   46          117         1.1351      
euclidean     all goals   46          136         1.4531      
chebyshev     all goals   46          155         1.5206      
zero          all goals   46          298         2.7197      


## 4. Display the comparison results

In [4]:
try:
    import pandas as pd
    results = pd.DataFrame(rows)
    display(results)
except ImportError:
    results = rows
    for row in results:
        print(row)

,heuristic,total_path_cost,nodes_expanded,execution_time_ms
0,manhattan,46,117,1.1351
1,euclidean,46,136,1.4531
2,chebyshev,46,155,1.5206
3,zero,46,298,2.7197


## 5. Save the results to CSV

In [5]:
output = ROOT / "heuristic_comparison.csv"
with output.open("w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=rows[0].keys())
    writer.writeheader()
    writer.writerows(rows)

print(f"Saved results to: {output}")
print("Note: execution time can vary between runs and computers.")

Saved results to: c:\Users\maitr\Downloads\Warehouse_Robot_AI_Project_GUI\Warehouse_Robot_AI_Project\heuristic_comparison.csv
Note: execution time can vary between runs and computers.


## 6. How to interpret the results
- **Total path cost:** total number of moves across all route segments.
- **Nodes expanded:** number of search nodes expanded by A*; fewer is generally more efficient.
- **Execution time:** measured runtime in milliseconds; it can vary across runs and machines.

The Zero heuristic behaves like uninformed uniform-cost search when movement costs are uniform. The experiment reports measured results rather than assuming which heuristic will perform best.